In [27]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack

from lightgbm import LGBMClassifier


In [28]:
# PATH 
df = pd.read_csv(r"C:\Users\Arka\OneDrive\Desktop\phishwd project\Dataset\archive\phishing_site_urls.csv")

# Check columns
print(df.columns)

Index(['URL', 'Label'], dtype='object')


In [29]:
# Make sure label is numeric
df["Label"] = df["Label"].map({
    "good": 0,
    "bad": 1
})

print(df["Label"].value_counts())
print(df["Label"].dtype)


Label
0    392924
1    156422
Name: count, dtype: int64
int64


In [30]:
# Add URL length feature
df["url_length"] = df["URL"].astype(str).apply(len)
print(df[["URL", "url_length"]].head())


                                                 URL  url_length
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...         225
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...          81
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....         177
3  mail.printakid.com/www.online.americanexpress....          60
4  thewhiskeydregs.com/wp-content/themes/widescre...         116


In [31]:
print(df.head())
print(df.isnull().sum())


                                                 URL  Label  url_length
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...      1         225
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...      1          81
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....      1         177
3  mail.printakid.com/www.online.americanexpress....      1          60
4  thewhiskeydregs.com/wp-content/themes/widescre...      1         116
URL           0
Label         0
url_length    0
dtype: int64


In [32]:
#url pre-processing 
df["URL"] = df["URL"].astype(str).str.lower().str.strip()


In [33]:
X = df["URL"]
y = df["Label"]
#Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import numpy as np
import re

# ----------- TF-IDF -----------
tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# ----------- Lexical Feature Functions -----------

def url_length(url):
    return len(url)

def digit_count(url):
    return sum(c.isdigit() for c in url)

def special_char_count(url):
    return len(re.findall(r'[^a-zA-Z0-9]', url))


# ----------- Apply Lexical Features -----------

X_train_lexical = np.array([
    [url_length(url), digit_count(url), special_char_count(url)]
    for url in X_train
])

X_test_lexical = np.array([
    [url_length(url), digit_count(url), special_char_count(url)]
    for url in X_test
])


# ----------- Combine (Hybrid Features) -----------

X_train_final = hstack([X_train_tfidf, X_train_lexical])
X_test_final = hstack([X_test_tfidf, X_test_lexical])


In [35]:
from scipy.sparse import hstack
import numpy as np
import re

def get_lexical_features(url_series):
    features = []
    for url in url_series:
        u_len = len(url)
        d_count = sum(c.isdigit() for c in url)
        s_count = len(re.findall(r'[^a-zA-Z0-9]', url))
        features.append([u_len, d_count, s_count])
    return np.array(features)

X_train_lexical = get_lexical_features(X_train)
X_test_lexical = get_lexical_features(X_test)

# Combine TF-IDF features with all 3 lexical features
X_train_final = hstack([X_train_tfidf, X_train_lexical])
X_test_final = hstack([X_test_tfidf, X_test_lexical])


In [36]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

lgbm_model.fit(X_train_final, y_train)



[LightGBM] [Info] Number of positive: 125137, number of negative: 314339
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 206.023297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3916161
[LightGBM] [Info] Number of data points in the train set: 439476, number of used features: 49982
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.284741 -> initscore=-0.921063
[LightGBM] [Info] Start training from score -0.921063


LGBMClassifier(learning_rate=0.05, n_estimators=300, objective='binary',
               random_state=42)

In [37]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = lgbm_model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


d:\Anaconda3lab\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


Accuracy: 0.9508237007372349

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.98      0.97     78585
           1       0.95      0.88      0.91     31285

    accuracy                           0.95    109870
   macro avg       0.95      0.93      0.94    109870
weighted avg       0.95      0.95      0.95    109870


Confusion Matrix:
 [[77035  1550]
 [ 3853 27432]]


In [38]:
import pickle

with open("phishlang_model.pkl", "wb") as f:
    pickle.dump(lgbm_model, f)

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print(" Model and TF-IDF saved successfully")


 Model and TF-IDF saved successfully


In [39]:
from scipy.sparse import hstack
import re

def predict_url(url, threshold=0.9):
    url = url.lower().strip()

    vec = tfidf.transform([url])
    u_len = len(url)
    d_count = sum(c.isdigit() for c in url)
    s_count = len(re.findall(r'[^a-zA-Z0-9]', url))
    lexical = [[u_len, d_count, s_count]]
    final_vec = hstack([vec, lexical])

    prob = lgbm_model.predict_proba(final_vec)[0][1]

    if prob >= threshold:
        return f"PHISHING ({prob:.2f})"
    else:
        return f"LEGIT ({prob:.2f})"

print(predict_url("http://secure-paypal-login.verify-user.com"))
print(predict_url("https://www.google.com"))
print(predict_url("https://www.facebook.com"))
print(predict_url("https://www.tryhackme.com"))


PHISHING (1.00)
LEGIT (0.83)
LEGIT (0.74)
LEGIT (0.82)


d:\Anaconda3lab\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")
